In [ ]:
import os, gc, time, numpy as np, pandas as pd, torch, timm, torch.nn as nn
import matplotlib.pyplot as plt
from PIL import Image
from concurrent.futures import ThreadPoolExecutor
from scipy.stats import rankdata
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_curve, f1_score, precision_score, recall_score
from tqdm import tqdm

LABELS = ["Cardiomegaly", "Edema", "Consolidation", "Atelectasis", "Pneumothorax", "Pleural Effusion"]
MODEL, EPOCHS, SEEDS, BS, LR = "densenet121", 7, [0, 1, 2], 64, 1e-4
OUT = r"G:\runs"; RD = rf"{OUT}\nih_train"; os.makedirs(RD, exist_ok=True)

plt.rcParams.update({"font.family": "serif", "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "mathtext.fontset": "stix", "font.size": 8, "axes.labelsize": 8, "axes.titlesize": 8,
    "xtick.labelsize": 7, "ytick.labelsize": 7, "legend.fontsize": 7, "axes.linewidth": 0.6,
    "lines.linewidth": 1.0, "grid.linewidth": 0.4, "figure.dpi": 150, "pdf.fonttype": 42})
COL1, COL2 = 3.5, 7.16
def save_fig(fig, name):
    fig.savefig(rf"{RD}\{name}.pdf", bbox_inches="tight")
    fig.savefig(rf"{RD}\{name}.png", dpi=600, bbox_inches="tight"); plt.show()

nih = pd.read_csv(r"G:\nih_clean.csv")
cx = pd.read_csv(r"G:\chexpert_final_clean.csv"); cxt = cx[cx.split == "test"].reset_index(drop=True)
arr = np.memmap(r"G:\nih_224.u8", dtype=np.uint8, mode="r", shape=(len(nih), 224, 224))
Yn = nih[LABELS].values.astype(int); Yc = cxt[LABELS].values.astype(int)

def load224(p):
    with Image.open(p) as im:
        return np.asarray(im.convert("L").resize((224, 224), Image.BILINEAR))
CXC = r"G:\cx_test_224.npy"
if os.path.exists(CXC):
    XC = np.load(CXC)
else:
    with ThreadPoolExecutor(12) as pool:
        XC = np.stack(list(tqdm(pool.map(load224, cxt.img_path.values), total=len(cxt), desc="caching CheXpert test")))
    np.save(CXC, XC)

MEAN = torch.tensor([0.485, 0.456, 0.406], device="cuda").view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225], device="cuda").view(1, 3, 1, 1)
def fauc(y, s):
    y = np.asarray(y).astype(bool); r = rankdata(s); n1 = y.sum(); n0 = len(y) - n1
    return (r[y].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)
print("GPU:", torch.cuda.get_device_name(0), "| NIH", len(nih), "| CheXpert test", XC.shape)

In [ ]:
tv = np.where(nih.official.values == "train_val")[0]; te_i = np.where(nih.official.values == "test")[0]
a, b = next(GroupShuffleSplit(1, test_size=0.1, random_state=42).split(tv, groups=nih.patient_id.values[tv]))
tr_i, va_i = np.sort(tv[a]), np.sort(tv[b])

pid = nih.patient_id.values
S = [set(pid[tr_i]), set(pid[va_i]), set(pid[te_i])]
assert not (S[0] & S[1]) and not (S[0] & S[2]) and not (S[1] & S[2]), "patient leakage!"
nih["split_rev"] = "train"; nih.loc[va_i, "split_rev"] = "val"; nih.loc[te_i, "split_rev"] = "test"
nih.to_csv(r"G:\nih_split.csv", index=False)
print("train/val/test images:", len(tr_i), len(va_i), len(te_i))
print("patients:", [len(x) for x in S])
print(pd.DataFrame({k: nih.loc[v, LABELS].mean() for k, v in [("train", tr_i), ("val", va_i), ("test", te_i)]}).round(3))

In [ ]:
def gpu_batch(A, idx, flip=False):
    x = torch.from_numpy(np.array(A[idx])).cuda(non_blocking=True).float().div(255).unsqueeze(1)
    if flip:
        f = torch.rand(len(idx), device="cuda") < 0.5
        x = torch.where(f.view(-1, 1, 1, 1), x.flip(-1), x)
    return (x.repeat(1, 3, 1, 1) - MEAN) / STD

@torch.no_grad()
def predict_idx(m, A, idx, bs=256):
    m.eval(); out = []
    for i in range(0, len(idx), bs):
        with torch.autocast("cuda"):
            o = m(gpu_batch(A, idx[i:i + bs]))
        out.append(torch.sigmoid(o.float()).cpu())
    return torch.cat(out).numpy()

def mean_auc(Y, P): return float(np.mean([fauc(Y[:, i], P[:, i]) for i in range(len(LABELS))]))
def bce(Y, P):
    P = np.clip(P, 1e-6, 1 - 1e-6); return float(-(Y * np.log(P) + (1 - Y) * np.log(1 - P)).mean())
lossf = nn.BCEWithLogitsLoss()

def train_seed(seed):
    tag = f"{MODEL}_nih_e{EPOCHS}_seed{seed}"
    BEST, CKPT, HIST = rf"{RD}\{tag}.pt", rf"{RD}\{tag}_last.pt", rf"{RD}\{tag}_history.csv"
    torch.manual_seed(seed); np.random.seed(seed)
    m = timm.create_model(MODEL, pretrained=True, num_classes=len(LABELS)).cuda()
    o = torch.optim.AdamW(m.parameters(), lr=LR, weight_decay=1e-4)
    spe = int(np.ceil(len(tr_i) / BS))
    sc = torch.optim.lr_scheduler.CosineAnnealingLR(o, T_max=EPOCHS * spe)
    sl = torch.amp.GradScaler("cuda")
    hist, best, start = [], 0.0, 0
    if os.path.exists(CKPT):
        ck = torch.load(CKPT, map_location="cuda", weights_only=False)
        m.load_state_dict(ck["model"]); o.load_state_dict(ck["opt"]); sc.load_state_dict(ck["sched"])
        sl.load_state_dict(ck["scaler"]); hist, best, start = ck["hist"], ck["best"], ck["ep"] + 1
        print(f"seed {seed}: resumed, next epoch {start + 1}")
    for ep in range(start, EPOCHS):
        perm = np.random.default_rng(seed * 100 + ep).permutation(tr_i); m.train(); t0 = time.time()
        rl, n = torch.zeros((), device="cuda"), 0
        for k in tqdm(range(spe), desc=f"seed {seed} epoch {ep + 1}/{EPOCHS}"):
            idx = np.sort(perm[k * BS:(k + 1) * BS])
            x = gpu_batch(arr, idx, flip=True); y = torch.from_numpy(Yn[idx]).cuda().float()
            o.zero_grad(set_to_none=True)
            with torch.autocast("cuda"):
                loss = lossf(m(x), y)
            sl.scale(loss).backward(); sl.step(o); sl.update(); sc.step()
            rl += loss.detach() * len(idx); n += len(idx)
        Pv = predict_idx(m, arr, va_i); Yv = Yn[va_i]
        row = dict(epoch=ep + 1, train_loss=float(rl) / n, val_loss=bce(Yv, Pv), val_auroc=mean_auc(Yv, Pv), sec=time.time() - t0)
        hist.append(row); pd.DataFrame(hist).to_csv(HIST, index=False)
        print(f"seed {seed} ep {ep + 1} | loss {row['train_loss']:.4f}/{row['val_loss']:.4f} | val AUROC {row['val_auroc']:.4f} | {row['sec'] / 60:.1f} min")
        if row["val_auroc"] > best:
            best = row["val_auroc"]; torch.save(m.state_dict(), BEST)
        torch.save(dict(model=m.state_dict(), opt=o.state_dict(), sched=sc.state_dict(), scaler=sl.state_dict(),
                        hist=hist, best=best, ep=ep), CKPT)
    m.load_state_dict(torch.load(BEST, map_location="cuda", weights_only=True))
    z = dict(nih_val_P=predict_idx(m, arr, va_i), nih_val_Y=Yn[va_i],
             nih_test_P=predict_idx(m, arr, te_i), nih_test_Y=Yn[te_i],
             cx_test_P=predict_idx(m, XC, np.arange(len(XC))), cx_test_Y=Yc)
    np.savez(rf"{RD}\{tag}_preds.npz", **z)
    print(f"seed {seed}: NIH test AUROC {mean_auc(z['nih_test_Y'], z['nih_test_P']):.4f} | "
          f"CheXpert (external) AUROC {mean_auc(z['cx_test_Y'], z['cx_test_P']):.4f}")
    del m, o, sc, sl; gc.collect(); torch.cuda.empty_cache()

In [ ]:
RUN = [0, 1, 2]           # first only seed 0; after checking it, use [1, 2]
for s in RUN:
    train_seed(s)

In [ ]:
SR = [s for s in SEEDS if os.path.exists(rf"{RD}\{MODEL}_nih_e{EPOCHS}_seed{s}_preds.npz")]
print("reverse-direction seeds found:", SR)
ZR = {s: np.load(rf"{RD}\{MODEL}_nih_e{EPOCHS}_seed{s}_preds.npz") for s in SR}
ZF = {s: np.load(rf"{OUT}\{MODEL}_e{EPOCHS}_seed{s}_results\preds.npz") for s in SR}
NPF = {s: np.load(rf"{OUT}\nih_eval\seed{s}_nih_preds.npy") for s in SR}
off = nih.official.values == "test"; Ynt = Yn[off]
assert (ZF[SR[0]]["test_Y"] == Yc).all() and (ZR[SR[0]]["nih_test_Y"] == Ynt).all()

pm = lambda x: f"{np.mean(x):.3f} ± {np.std(x, ddof=1) if len(x) > 1 else 0:.3f}"
M = {("CheXpert", "CheXpert test"): [mean_auc(Yc, ZF[s]["test_P"]) for s in SR],
     ("CheXpert", "NIH test"): [mean_auc(Ynt, NPF[s][off]) for s in SR],
     ("NIH", "CheXpert test"): [mean_auc(Yc, ZR[s]["cx_test_P"]) for s in SR],
     ("NIH", "NIH test"): [mean_auc(Ynt, ZR[s]["nih_test_P"]) for s in SR]}
mat = pd.DataFrame({te: {tr: pm(M[(tr, te)]) for tr in ["CheXpert", "NIH"]} for te in ["CheXpert test", "NIH test"]})
mat.index.name = "Trained on"; display(mat); mat.to_csv(rf"{RD}\table_transfer_matrix.csv")

fig, ax = plt.subplots(figsize=(COL1 * 1.1, 2.4))
V = np.array([[np.mean(M[(tr, te)]) for te in ["CheXpert test", "NIH test"]] for tr in ["CheXpert", "NIH"]])
im = ax.imshow(V, cmap="Greys", vmin=0.7, vmax=0.9)
for i in range(2):
    for j in range(2):
        ax.text(j, i, f"{V[i, j]:.3f}", ha="center", va="center", color="white" if V[i, j] > 0.82 else "black")
ax.set_xticks([0, 1]); ax.set_xticklabels(["CheXpert test", "NIH test"]); ax.set_yticks([0, 1]); ax.set_yticklabels(["CheXpert", "NIH"])
ax.set(xlabel="Tested on", ylabel="Trained on"); ax.grid(False)
fig.tight_layout(); save_fig(fig, "fig_transfer_matrix")

fig, ax = plt.subplots(1, 2, figsize=(COL2, 2.3))
for s in SR:
    h = pd.read_csv(rf"{RD}\{MODEL}_nih_e{EPOCHS}_seed{s}_history.csv")
    ax[0].plot(h.epoch, h.train_loss, "-", color=f"C{s}", label=f"Train, seed {s}"); ax[0].plot(h.epoch, h.val_loss, "--", color=f"C{s}")
    ax[1].plot(h.epoch, h.val_auroc, "-o", ms=2.5, color=f"C{s}", label=f"Seed {s}")
ax[0].set(xlabel="Epoch", ylabel="BCE loss (dashed = validation)"); ax[1].set(xlabel="Epoch", ylabel="NIH validation mean AUROC")
for a in ax: a.grid(True, alpha=0.3); a.legend(frameon=False, fontsize=6)
fig.tight_layout(); save_fig(fig, "fig_nih_training_curves")

In [ ]:
B = 300
def groups(p): return list(pd.Series(np.arange(len(p))).groupby(np.asarray(p)).apply(np.array).values)
gc_, gn_ = groups(cxt.patient_id.values), groups(nih.patient_id.values[off])
Pcf = {s: ZF[s]["test_P"] for s in SR}; Pnf = {s: NPF[s][off] for s in SR}
Pnr = {s: ZR[s]["nih_test_P"] for s in SR}; Pcr = {s: ZR[s]["cx_test_P"] for s in SR}
L = len(LABELS); rng = np.random.default_rng(0)
bf, br, b_in, b_out = np.zeros((B, L)), np.zeros((B, L)), np.zeros((B, L)), np.zeros((B, L))
avg = lambda Y, Ps, ix, i: np.mean([fauc(Y[ix, i], Ps[s][ix, i]) for s in SR])
for r_ in tqdm(range(B), desc="bootstrap"):
    ic = np.concatenate([gc_[k] for k in rng.integers(0, len(gc_), len(gc_))])
    iN = np.concatenate([gn_[k] for k in rng.integers(0, len(gn_), len(gn_))])
    for i in range(L):
        cf, nf, nr, cr = avg(Yc, Pcf, ic, i), avg(Ynt, Pnf, iN, i), avg(Ynt, Pnr, iN, i), avg(Yc, Pcr, ic, i)
        bf[r_, i], br[r_, i], b_in[r_, i], b_out[r_, i] = cf - nf, nr - cr, nr, cr
full = lambda Y, Ps: np.array([avg(Y, Ps, np.arange(len(Y)), i) for i in range(L)])
cf_, nf_, nr_, cr_ = full(Yc, Pcf), full(Ynt, Pnf), full(Ynt, Pnr), full(Yc, Pcr)
fw, rv = cf_ - nf_, nr_ - cr_
ci = lambda a: np.percentile(a, [2.5, 97.5], axis=0)
cf_ci, cr_ci = ci(bf), ci(br)
rows = [{"Label": l, "NIH-trained: NIH test": nr_[i], "NIH-trained: CheXpert": cr_[i], "Reverse gap": rv[i],
         "Reverse CI": f"[{cr_ci[0, i]:.3f}, {cr_ci[1, i]:.3f}]", "Forward gap": fw[i],
         "Forward CI": f"[{cf_ci[0, i]:.3f}, {cf_ci[1, i]:.3f}]"} for i, l in enumerate(LABELS)]
mlo, mhi = np.percentile(br.mean(1), [2.5, 97.5]); flo, fhi = np.percentile(bf.mean(1), [2.5, 97.5])
rows.append({"Label": "Mean", "NIH-trained: NIH test": nr_.mean(), "NIH-trained: CheXpert": cr_.mean(), "Reverse gap": rv.mean(),
             "Reverse CI": f"[{mlo:.3f}, {mhi:.3f}]", "Forward gap": fw.mean(), "Forward CI": f"[{flo:.3f}, {fhi:.3f}]"})
gaps = pd.DataFrame(rows); display(gaps.round(3)); gaps.to_csv(rf"{RD}\table_gaps_both_directions.csv", index=False)

x = np.arange(L); clip = lambda a: np.clip(a, 0, None)
fig, ax = plt.subplots(figsize=(COL1 * 1.6, 2.7))
ax.bar(x - 0.2, fw, 0.4, yerr=[clip(fw - cf_ci[0]), clip(cf_ci[1] - fw)], capsize=2, color="0.35", label="CheXpert → NIH")
ax.bar(x + 0.2, rv, 0.4, yerr=[clip(rv - cr_ci[0]), clip(cr_ci[1] - rv)], capsize=2, color="0.75", edgecolor="k", lw=0.4, label="NIH → CheXpert")
ax.axhline(0, color="k", lw=0.6); ax.set_xticks(x); ax.set_xticklabels(LABELS, rotation=35, ha="right")
ax.set(ylabel="AUROC drop (in-hospital − external)"); ax.legend(frameon=False); ax.grid(True, axis="y", alpha=0.3)
fig.tight_layout(); save_fig(fig, "fig_gap_both_directions")

fig, ax = plt.subplots(figsize=(COL1 * 1.5, 2.7)); ci_i, ci_o = ci(b_in), ci(b_out)
ax.bar(x - 0.2, nr_, 0.4, yerr=[clip(nr_ - ci_i[0]), clip(ci_i[1] - nr_)], capsize=2, color="0.35", label="NIH test (in-hospital)")
ax.bar(x + 0.2, cr_, 0.4, yerr=[clip(cr_ - ci_o[0]), clip(ci_o[1] - cr_)], capsize=2, color="0.75", edgecolor="k", lw=0.4, label="CheXpert (external)")
ax.set_xticks(x); ax.set_xticklabels(LABELS, rotation=35, ha="right"); ax.set(ylabel="AUROC", ylim=(0.5, 0.95))
ax.legend(frameon=False); ax.grid(True, axis="y", alpha=0.3)
fig.tight_layout(); save_fig(fig, "fig_auroc_nih_trained")

In [ ]:
s0 = SR[0]; Z0 = ZR[s0]
def reliab(y, p, nb=10):
    bi = np.clip((p * nb).astype(int), 0, nb - 1); xs, ys, ws = [], [], []
    for k in range(nb):
        mk = bi == k
        if mk.sum() > 20: xs.append(p[mk].mean()); ys.append(y[mk].mean()); ws.append(mk.mean())
    return np.array(xs), np.array(ys), np.array(ws)
fig, axes = plt.subplots(2, 3, figsize=(COL2, 4.2)); ece = []
for i, (a, l) in enumerate(zip(axes.ravel(), LABELS)):
    row = {"Label": l}
    for nm, Y_, P_, st in [("NIH test", Z0["nih_test_Y"], Z0["nih_test_P"], "-o"), ("CheXpert", Z0["cx_test_Y"], Z0["cx_test_P"], "--s")]:
        xs, ys, ws = reliab(Y_[:, i], P_[:, i]); a.plot(xs, ys, st, ms=2.5, label=nm); row["ECE " + nm] = float((ws * np.abs(xs - ys)).sum())
    ece.append(row); a.plot([0, 1], [0, 1], "k:", lw=0.6); a.set_title(l); a.grid(True, alpha=0.3)
    a.set(xlabel="Predicted probability", ylabel="Observed frequency"); a.legend(frameon=False, fontsize=6)
fig.tight_layout(); save_fig(fig, "fig_calibration_nih_trained")
ece = pd.DataFrame(ece); display(ece.round(3)); ece.to_csv(rf"{RD}\table_ece_nih_trained.csv", index=False)

grid = np.linspace(0.02, 0.95, 94)
thr = np.array([grid[int(np.argmax([f1_score(Z0["nih_val_Y"][:, i], Z0["nih_val_P"][:, i] >= t, zero_division=0) for t in grid]))] for i in range(L)])
rows = []
for i, l in enumerate(LABELS):
    for nm, Y_, P_ in [("NIH test", Z0["nih_test_Y"], Z0["nih_test_P"]), ("CheXpert", Z0["cx_test_Y"], Z0["cx_test_P"])]:
        pr = (P_[:, i] >= thr[i]).astype(int)
        rows.append({"Label": l, "Set": nm, "Prevalence": Y_[:, i].mean(), "Precision": precision_score(Y_[:, i], pr, zero_division=0),
                     "Recall": recall_score(Y_[:, i], pr, zero_division=0), "F1": f1_score(Y_[:, i], pr, zero_division=0)})
tt = pd.DataFrame(rows); display(tt.round(3)); tt.to_csv(rf"{RD}\table_fixed_threshold_nih_trained.csv", index=False)

In [ ]:
SR = [s for s in SEEDS if os.path.exists(rf"{RD}\{MODEL}_nih_e{EPOCHS}_seed{s}_preds.npz")]
print("seeds found (NIH-trained):", SR)
ZR = {s: np.load(rf"{RD}\{MODEL}_nih_e{EPOCHS}_seed{s}_preds.npz") for s in SR}
ZF = {s: np.load(rf"{OUT}\{MODEL}_e{EPOCHS}_seed{s}_results\preds.npz") for s in SR}
NPF = {s: np.load(rf"{OUT}\nih_eval\seed{s}_nih_preds.npy") for s in SR}
off = nih.official.values == "test"; Ynt = Yn[off]
assert (ZF[SR[0]]["test_Y"] == Yc).all() and (ZR[SR[0]]["nih_test_Y"] == Ynt).all()

DIRS = {"CheXpert→CheXpert": (Yc, {s: ZF[s]["test_P"] for s in SR}),
        "CheXpert→NIH": (Ynt, {s: NPF[s][off] for s in SR}),
        "NIH→NIH": (Ynt, {s: ZR[s]["nih_test_P"] for s in SR}),
        "NIH→CheXpert": (Yc, {s: ZR[s]["cx_test_P"] for s in SR})}
PER = {d: np.array([[fauc(Y[:, i], Ps[s][:, i]) for i in range(L)] for s in SR]) for d, (Y, Ps) in DIRS.items()}
sd = lambda a, ax=0: a.std(ax, ddof=1) if a.shape[ax] > 1 else np.zeros(a.shape[1 - ax])
pmf = lambda m, s_: f"{m:.3f} ± {s_:.3f}"

# 1) the 2x2 transfer matrix (mean AUROC over the 6 labels, mean ± std over seeds)
mv = lambda d: PER[d].mean(1)
mat = pd.DataFrame({"CheXpert test": {"CheXpert": pmf(mv("CheXpert→CheXpert").mean(), mv("CheXpert→CheXpert").std(ddof=1) if len(SR) > 1 else 0),
                                     "NIH": pmf(mv("NIH→CheXpert").mean(), mv("NIH→CheXpert").std(ddof=1) if len(SR) > 1 else 0)},
                    "NIH test": {"CheXpert": pmf(mv("CheXpert→NIH").mean(), mv("CheXpert→NIH").std(ddof=1) if len(SR) > 1 else 0),
                                 "NIH": pmf(mv("NIH→NIH").mean(), mv("NIH→NIH").std(ddof=1) if len(SR) > 1 else 0)}})
mat.index.name = "Trained on"; display(mat); mat.to_csv(rf"{RD}\table_transfer_matrix.csv")

V = np.array([[mv("CheXpert→CheXpert").mean(), mv("CheXpert→NIH").mean()], [mv("NIH→CheXpert").mean(), mv("NIH→NIH").mean()]])
fig, ax = plt.subplots(figsize=(COL1 * 1.1, 2.4))
ax.imshow(V, cmap="Greys", vmin=V.min() - 0.03, vmax=V.max() + 0.03)
for i in range(2):
    for j in range(2):
        ax.text(j, i, f"{V[i, j]:.3f}", ha="center", va="center", color="white" if V[i, j] > V.mean() else "black")
ax.set_xticks([0, 1]); ax.set_xticklabels(["CheXpert test", "NIH test"]); ax.set_yticks([0, 1]); ax.set_yticklabels(["CheXpert", "NIH"])
ax.set(xlabel="Tested on", ylabel="Trained on"); ax.grid(False)
fig.tight_layout(); save_fig(fig, "fig_transfer_matrix")

# 2) per-label AUROC for all four directions
rows = {d: [pmf(PER[d][:, i].mean(), sd(PER[d])[i]) for i in range(L)] + [pmf(mv(d).mean(), mv(d).std(ddof=1) if len(SR) > 1 else 0)] for d in DIRS}
perlab = pd.DataFrame(rows, index=LABELS + ["Mean"]).T; perlab.index.name = "Direction"
display(perlab); perlab.to_csv(rf"{RD}\table_four_directions_per_label.csv")

Mx = np.array([PER[d].mean(0) for d in DIRS]); Mx = np.c_[Mx, Mx.mean(1)]
fig, ax = plt.subplots(figsize=(COL2, 2.6))
im = ax.imshow(Mx, cmap="Greys", vmin=Mx.min() - 0.03, vmax=Mx.max() + 0.03, aspect="auto")
for i in range(Mx.shape[0]):
    for j in range(Mx.shape[1]):
        ax.text(j, i, f"{Mx[i, j]:.3f}", ha="center", va="center", fontsize=7, color="white" if Mx[i, j] > Mx.mean() else "black")
ax.set_xticks(range(L + 1)); ax.set_xticklabels(LABELS + ["Mean"], rotation=30, ha="right")
ax.set_yticks(range(4)); ax.set_yticklabels(list(DIRS)); ax.grid(False)
fig.tight_layout(); save_fig(fig, "fig_four_directions_heatmap")

fig, ax = plt.subplots(figsize=(COL2, 2.8)); x = np.arange(L); w = 0.2
for k, (d, st) in enumerate(zip(DIRS, ["0.2", "0.45", "0.7", "0.9"])):
    ax.bar(x + (k - 1.5) * w, PER[d].mean(0), w, yerr=sd(PER[d]), capsize=1.5, color=st, edgecolor="k", lw=0.4, label=d)
ax.set_xticks(x); ax.set_xticklabels(LABELS, rotation=25, ha="right"); ax.set(ylabel="AUROC", ylim=(0.5, 0.97))
ax.legend(frameon=False, ncol=2, fontsize=6); ax.grid(True, axis="y", alpha=0.3)
fig.tight_layout(); save_fig(fig, "fig_four_directions_bars")

fig, ax = plt.subplots(1, 2, figsize=(COL2, 2.3))
for s in SR:
    h = pd.read_csv(rf"{RD}\{MODEL}_nih_e{EPOCHS}_seed{s}_history.csv")
    ax[0].plot(h.epoch, h.train_loss, "-", color=f"C{s}", label=f"Train, seed {s}"); ax[0].plot(h.epoch, h.val_loss, "--", color=f"C{s}")
    ax[1].plot(h.epoch, h.val_auroc, "-o", ms=2.5, color=f"C{s}", label=f"Seed {s}")
ax[0].set(xlabel="Epoch", ylabel="BCE loss (dashed = validation)"); ax[1].set(xlabel="Epoch", ylabel="NIH validation mean AUROC")
for a_ in ax: a_.grid(True, alpha=0.3); a_.legend(frameon=False, fontsize=6)
fig.tight_layout(); save_fig(fig, "fig_nih_training_curves")

# decision thresholds: chosen on each training hospital's own validation set
grid = np.linspace(0.02, 0.95, 94)
def best_thr(y, p): return grid[int(np.argmax([f1_score(y, p >= t, zero_division=0) for t in grid]))]
THR = {"CheXpert": {s: np.load(rf"{OUT}\{MODEL}_e{EPOCHS}_seed{s}_results\thresholds.npy") for s in SR},
       "NIH": {s: np.array([best_thr(ZR[s]["nih_val_Y"][:, i], ZR[s]["nih_val_P"][:, i]) for i in range(L)]) for s in SR}}

In [ ]:
B = 300
def groups(p): return list(pd.Series(np.arange(len(p))).groupby(np.asarray(p)).apply(np.array).values)
gc_, gn_ = groups(cxt.patient_id.values), groups(nih.patient_id.values[off])
rng = np.random.default_rng(0)
bf, br = np.zeros((B, L)), np.zeros((B, L)); b_in, b_out = np.zeros((B, L)), np.zeros((B, L))
avg = lambda d, ix, i: np.mean([fauc(DIRS[d][0][ix, i], DIRS[d][1][s][ix, i]) for s in SR])
for r_ in tqdm(range(B), desc="bootstrap"):
    ic = np.concatenate([gc_[k] for k in rng.integers(0, len(gc_), len(gc_))])
    iN = np.concatenate([gn_[k] for k in rng.integers(0, len(gn_), len(gn_))])
    for i in range(L):
        cc, cn, nn_, nc = avg("CheXpert→CheXpert", ic, i), avg("CheXpert→NIH", iN, i), avg("NIH→NIH", iN, i), avg("NIH→CheXpert", ic, i)
        bf[r_, i], br[r_, i], b_in[r_, i], b_out[r_, i] = cc - cn, nn_ - nc, nn_, nc
allx = lambda d: np.array([avg(d, np.arange(len(DIRS[d][0])), i) for i in range(L)])
fw = allx("CheXpert→CheXpert") - allx("CheXpert→NIH"); rv = allx("NIH→NIH") - allx("NIH→CheXpert")
ci = lambda a: np.percentile(a, [2.5, 97.5], axis=0)
cf_ci, cr_ci, as_ci = ci(bf), ci(br), ci(br - bf)
fmt = lambda c, i: f"[{c[0, i]:.3f}, {c[1, i]:.3f}]"
rows = [{"Label": l, "Forward gap (CheXpert→NIH)": fw[i], "Forward CI": fmt(cf_ci, i),
         "Reverse gap (NIH→CheXpert)": rv[i], "Reverse CI": fmt(cr_ci, i),
         "Asymmetry (rev − fwd)": rv[i] - fw[i], "Asymmetry CI": fmt(as_ci, i)} for i, l in enumerate(LABELS)]
ml = lambda a: np.percentile(a.mean(1), [2.5, 97.5])
rows.append({"Label": "Mean", "Forward gap (CheXpert→NIH)": fw.mean(), "Forward CI": "[%.3f, %.3f]" % tuple(ml(bf)),
             "Reverse gap (NIH→CheXpert)": rv.mean(), "Reverse CI": "[%.3f, %.3f]" % tuple(ml(br)),
             "Asymmetry (rev − fwd)": (rv - fw).mean(), "Asymmetry CI": "[%.3f, %.3f]" % tuple(ml(br - bf))})
gaps = pd.DataFrame(rows); display(gaps.round(3)); gaps.to_csv(rf"{RD}\table_gaps_both_directions.csv", index=False)

clip = lambda a: np.clip(a, 0, None); x = np.arange(L)
fig, ax = plt.subplots(figsize=(COL1 * 1.6, 2.7))
ax.bar(x - 0.2, fw, 0.4, yerr=[clip(fw - cf_ci[0]), clip(cf_ci[1] - fw)], capsize=2, color="0.35", label="CheXpert → NIH")
ax.bar(x + 0.2, rv, 0.4, yerr=[clip(rv - cr_ci[0]), clip(cr_ci[1] - rv)], capsize=2, color="0.75", edgecolor="k", lw=0.4, label="NIH → CheXpert")
ax.axhline(0, color="k", lw=0.6); ax.set_xticks(x); ax.set_xticklabels(LABELS, rotation=35, ha="right")
ax.set(ylabel="AUROC drop (in-hospital − external)"); ax.legend(frameon=False); ax.grid(True, axis="y", alpha=0.3)
fig.tight_layout(); save_fig(fig, "fig_gap_both_directions")

In [ ]:
def ece_of(y, p, nb=10):
    bi = np.clip((p * nb).astype(int), 0, nb - 1); e = 0.0
    for k in range(nb):
        mk = bi == k
        if mk.sum() > 0: e += mk.mean() * abs(p[mk].mean() - y[mk].mean())
    return e
ECE = {d: np.array([[ece_of(Y[:, i], Ps[s][:, i]) for i in range(L)] for s in SR]) for d, (Y, Ps) in DIRS.items()}
ece_tab = pd.DataFrame({d: [pmf(ECE[d][:, i].mean(), sd(ECE[d])[i]) for i in range(L)] + [pmf(ECE[d].mean(1).mean(), ECE[d].mean(1).std(ddof=1) if len(SR) > 1 else 0)]
                        for d in DIRS}, index=LABELS + ["Mean"]).T
ece_tab.index.name = "Direction (ECE, lower is better)"; display(ece_tab); ece_tab.to_csv(rf"{RD}\table_ece_four_directions.csv")

def reliab(y, p, nb=10):
    bi = np.clip((p * nb).astype(int), 0, nb - 1); xs, ys = [], []
    for k in range(nb):
        mk = bi == k
        if mk.sum() > 20: xs.append(p[mk].mean()); ys.append(y[mk].mean())
    return xs, ys
sty = {"CheXpert→CheXpert": "-o", "CheXpert→NIH": "--s", "NIH→NIH": "-^", "NIH→CheXpert": "--v"}
s0 = SR[0]
fig, axes = plt.subplots(2, 3, figsize=(COL2, 4.3))
for i, (a_, l) in enumerate(zip(axes.ravel(), LABELS)):
    for d, (Y, Ps) in DIRS.items():
        xs, ys = reliab(Y[:, i], Ps[s0][:, i]); a_.plot(xs, ys, sty[d], ms=2.5, label=d)
    a_.plot([0, 1], [0, 1], "k:", lw=0.6); a_.set_title(l); a_.grid(True, alpha=0.3)
    a_.set(xlabel="Predicted probability", ylabel="Observed frequency")
axes[0, 0].legend(frameon=False, fontsize=5.5)
fig.tight_layout(); save_fig(fig, "fig_calibration_four_directions")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(COL2, 4.3)); lst = {"CheXpert→CheXpert": "-", "CheXpert→NIH": "--", "NIH→NIH": "-.", "NIH→CheXpert": ":"}
for i, (a_, l) in enumerate(zip(axes.ravel(), LABELS)):
    for d, (Y, Ps) in DIRS.items():
        f_, t_, _ = roc_curve(Y[:, i], Ps[s0][:, i]); a_.plot(f_, t_, lst[d], label=f"{d} ({PER[d][SR.index(s0), i]:.3f})")
    a_.plot([0, 1], [0, 1], "k:", lw=0.5); a_.set_title(l); a_.grid(True, alpha=0.3)
    a_.set(xlabel="False positive rate", ylabel="True positive rate"); a_.legend(loc="lower right", frameon=False, fontsize=5)
fig.tight_layout(); save_fig(fig, "fig_roc_four_directions")

rows = []
for d, (Y, Ps) in DIRS.items():
    src_ = d.split("→")[0]
    for i, l in enumerate(LABELS):
        pr_, rc_, f1_ = [], [], []
        for s in SR:
            yh = (Ps[s][:, i] >= THR[src_][s][i]).astype(int)
            pr_.append(precision_score(Y[:, i], yh, zero_division=0)); rc_.append(recall_score(Y[:, i], yh, zero_division=0)); f1_.append(f1_score(Y[:, i], yh, zero_division=0))
        rows.append({"Direction": d, "Label": l, "Prevalence": Y[:, i].mean(), "Precision": np.mean(pr_), "Recall": np.mean(rc_), "F1": np.mean(f1_)})
thr_tab = pd.DataFrame(rows); display(thr_tab.pivot(index="Label", columns="Direction", values="F1").loc[LABELS][list(DIRS)].round(3))
thr_tab.to_csv(rf"{RD}\table_fixed_threshold_four_directions.csv", index=False)

In [ ]:
def prep(a):
    x = torch.from_numpy(np.array(a)).cuda().float().div(255).unsqueeze(1).repeat(1, 3, 1, 1)
    return (x - MEAN) / STD
def load_model(path):
    m = timm.create_model(MODEL, pretrained=False, num_classes=L).cuda()
    m.load_state_dict(torch.load(path, map_location="cuda", weights_only=True)); return m.eval()
def gradcam(model, a, cls):
    fm_ = {}
    h = model.features.register_forward_hook(lambda m, i, o: fm_.__setitem__("o", o))
    model.zero_grad(); out = model(prep(a[None])); h.remove()
    fm = fm_["o"]; g = torch.autograd.grad(out[0, cls], fm)[0]
    cam = torch.relu((g.mean((2, 3), keepdim=True) * fm).sum(1, keepdim=True))
    cam = torch.nn.functional.interpolate(cam, size=(224, 224), mode="bilinear", align_corners=False)[0, 0]
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    return cam.detach().cpu().numpy()

s0 = SR[0]
mN = load_model(rf"{RD}\{MODEL}_nih_e{EPOCHS}_seed{s0}.pt")        # NIH-trained
mC = load_model(rf"{OUT}\{MODEL}_e{EPOCHS}_seed{s0}.pt")            # CheXpert-trained (used in later steps)

LBL_NAME = "Pleural Effusion"        # change to "Cardiomegaly", "Edema", ... and rerun Steps 9 and 10
LBL = LABELS.index(LBL_NAME); thrN = THR["NIH"][s0][LBL]
Pn_, Pc_ = ZR[s0]["nih_test_P"][:, LBL], ZR[s0]["cx_test_P"][:, LBL]
Yn_, Yc_ = Ynt[:, LBL], Yc[:, LBL]
GR = {"NIH test, correct": ("nih", np.where((Yn_ == 1) & (Pn_ >= thrN))[0]),
      "CheXpert, correct": ("cx", np.where((Yc_ == 1) & (Pc_ >= thrN))[0]),
      "CheXpert, missed": ("cx", np.where((Yc_ == 1) & (Pc_ < thrN))[0])}
print({k: len(v[1]) for k, v in GR.items()}, "images available per group")
img_of = lambda src_, i: np.asarray(arr[te_i[i]]) if src_ == "nih" else XC[i]

rng = np.random.default_rng(3); K = 4
fig, ax = plt.subplots(len(GR), K, figsize=(COL2, 5.2))
for r, (nm, (src_, ids)) in enumerate(GR.items()):
    pick = rng.choice(ids, min(K, len(ids)), replace=False)
    for c in range(K):
        ax[r, c].axis("off")
        if c < len(pick):
            im_ = img_of(src_, pick[c]); cam = gradcam(mN, im_, LBL)
            ax[r, c].imshow(im_, cmap="gray"); ax[r, c].imshow(cam, cmap="jet", alpha=0.4)
    ax[r, 0].set_title(nm, loc="left", fontsize=7)
fig.suptitle(f"Grad-CAM, NIH-trained model, {LBL_NAME}", fontsize=8)
fig.tight_layout(); save_fig(fig, f"fig_gradcam_nih_trained_{LBL_NAME.replace(' ', '_')}")

In [ ]:
from scipy.stats import mannwhitneyu
MRG = int(0.15 * 224)
edge = np.ones((224, 224), bool); edge[MRG:-MRG, MRG:-MRG] = False
bottom = np.zeros((224, 224), bool); bottom[149:, :] = True
rng = np.random.default_rng(7); K2 = 200
stat, rows = {}, []
for nm, (src_, ids) in GR.items():
    pick = rng.choice(ids, min(K2, len(ids)), replace=False); ef, bf_, zero = [], [], 0
    for i in tqdm(pick, desc=nm, leave=False):
        cam = gradcam(mN, img_of(src_, i), LBL); s_ = cam.sum()
        if s_ <= 0: zero += 1; continue
        ef.append(cam[edge].sum() / s_); bf_.append(cam[bottom].sum() / s_)
    ef, bf_ = np.array(ef), np.array(bf_); stat[nm] = (ef, bf_)
    f = lambda a: f"{a.mean():.3f} ± {a.std(ddof=1) / np.sqrt(len(a)):.3f}" if len(a) > 1 else "n/a"
    rows.append({"Group": nm, "Images": len(pick), "Empty heatmaps": zero, "Edge fraction": f(ef), "Bottom-third fraction": f(bf_)})
cam_tab = pd.DataFrame(rows).set_index("Group"); display(cam_tab)
print("Uniform-heat baseline: edge 0.50, bottom third 0.33")
ks = list(stat)
for j, nm in enumerate(["Edge fraction", "Bottom-third fraction"]):
    for k1, k2 in [(ks[0], ks[1]), (ks[1], ks[2])]:
        if len(stat[k1][j]) > 1 and len(stat[k2][j]) > 1:
            print(f"{nm}: {k1} vs {k2}, Mann-Whitney p = {mannwhitneyu(stat[k1][j], stat[k2][j]).pvalue:.2g}")
cam_tab.to_csv(rf"{RD}\table_gradcam_regions_nih_trained.csv")
fig, ax = plt.subplots(1, 2, figsize=(COL2, 2.3))
for j, (nm, base) in enumerate([("Edge fraction", 0.50), ("Bottom-third fraction", 0.33)]):
    keep = [k for k in ks if len(stat[k][j])]
    ax[j].boxplot([stat[k][j] for k in keep], labels=[k.replace(", ", "\n") for k in keep], showfliers=False)
    ax[j].axhline(base, color="k", ls=":", lw=0.6); ax[j].set(ylabel=nm); ax[j].grid(True, axis="y", alpha=0.3)
fig.tight_layout(); save_fig(fig, "fig_gradcam_regions_nih_trained")

In [ ]:
@torch.no_grad()
def masked_pred(model, A, idx, frac, bs=256):
    mg = int(224 * frac); out = []
    for i in range(0, len(idx), bs):
        x = torch.from_numpy(np.array(A[idx[i:i + bs]])).cuda().float()
        if mg > 0:
            fill = x.flatten(1).median(1).values.view(-1, 1, 1)
            x[:, :mg, :] = fill; x[:, -mg:, :] = fill; x[:, :, :mg] = fill; x[:, :, -mg:] = fill
        x = (x.div(255).unsqueeze(1).repeat(1, 3, 1, 1) - MEAN) / STD
        with torch.autocast("cuda"):
            o = model(x)
        out.append(torch.sigmoid(o.float()).cpu())
    return torch.cat(out).numpy()

TESTS = {"CheXpert test": (XC, np.arange(len(XC)), Yc), "NIH test": (arr, te_i, Ynt)}
rows = []
for mn, mod in [("CheXpert-trained", mC), ("NIH-trained", mN)]:
    for tn, (A, idx, Y) in TESTS.items():
        for fr in [0, 0.05, 0.10, 0.15]:
            P = masked_pred(mod, A, idx, fr)
            rows.append({"Model": mn, "Test set": tn, "Margin masked": f"{int(fr * 100)}%", "Mean AUROC": mean_auc(Y, P)})
mt = pd.DataFrame(rows)
pv = mt.pivot_table(index=["Model", "Test set"], columns="Margin masked", values="Mean AUROC", sort=False)
pv = pv[["0%", "5%", "10%", "15%"]]; chg = pv.sub(pv["0%"], axis=0); chg.columns = ["change " + c for c in chg.columns]
display(pv.round(4)); display(chg.round(4)); mt.to_csv(rf"{RD}\table_border_masking_both_models.csv", index=False)
fig, ax = plt.subplots(figsize=(COL1 * 1.3, 2.6))
for (mn, tn), r in pv.iterrows():
    ax.plot(range(4), r.values, "-o" if "NIH-trained" in mn else "--s", ms=3, label=f"{mn} on {tn}")
ax.set_xticks(range(4)); ax.set_xticklabels(pv.columns); ax.set(xlabel="Border masked", ylabel="Mean AUROC")
ax.legend(frameon=False, fontsize=5.5); ax.grid(True, alpha=0.3)
fig.tight_layout(); save_fig(fig, "fig_border_masking_both_models")

In [ ]:
def masks(df, view=False):
    g = {"Male": df.Sex == "Male", "Female": df.Sex == "Female", "Age <50": df.Age < 50,
         "Age 50-65": df.Age.between(50, 65), "Age >65": df.Age > 65}
    if view: g["PA view"] = df["view"] == "PA"; g["AP view"] = df["view"] == "AP"
    return {k: v.values for k, v in g.items()}
def safe_auc(y, p): return fauc(y, p) if 0 < y.sum() < len(y) else np.nan
def group_auc(Y, Ps, m):
    return np.nanmean([np.nanmean([safe_auc(Y[m, i], Ps[s][m, i]) for s in SR]) for i in range(L)])
mc, mn_ = masks(cxt), masks(nih[off].reset_index(drop=True), view=True)
t_cx = pd.DataFrame({d: {g: group_auc(DIRS[d][0], DIRS[d][1], m) for g, m in mc.items()} for d in ["CheXpert→CheXpert", "NIH→CheXpert"]})
t_nh = pd.DataFrame({d: {g: group_auc(DIRS[d][0], DIRS[d][1], m) for g, m in mn_.items()} for d in ["NIH→NIH", "CheXpert→NIH"]})
t_cx.insert(0, "N", [int(m.sum()) for m in mc.values()]); t_nh.insert(0, "N", [int(m.sum()) for m in mn_.values()])
t_cx.index.name = "CheXpert test groups"; t_nh.index.name = "NIH test groups"
display(t_cx.round(3)); display(t_nh.round(3))
t_cx.to_csv(rf"{RD}\table_subgroups_cx_test.csv"); t_nh.to_csv(rf"{RD}\table_subgroups_nih_test.csv")
fig, ax = plt.subplots(1, 2, figsize=(COL2, 2.6))
for a_, t_, ttl in [(ax[0], t_cx, "CheXpert test set"), (ax[1], t_nh, "NIH test set")]:
    cols = [c for c in t_.columns if c != "N"]; xx = np.arange(len(t_)); w = 0.38
    for k, c in enumerate(cols):
        a_.bar(xx + (k - 0.5) * w, t_[c], w, color=["0.3", "0.75"][k], edgecolor="k", lw=0.4, label=c)
    a_.set_xticks(xx); a_.set_xticklabels(t_.index, rotation=40, ha="right"); a_.set(ylabel="Mean AUROC", ylim=(0.6, 0.95), title=ttl)
    a_.legend(frameon=False, fontsize=6); a_.grid(True, axis="y", alpha=0.3)
fig.tight_layout(); save_fig(fig, "fig_subgroups_both_directions")

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import roc_auc_score

NS = 1500; nt = nih[off].reset_index(drop=True)
ci_ = cxt.drop_duplicates("patient_id").sample(NS, random_state=1).index.values
ni_ = nt.drop_duplicates("patient_id").sample(NS, random_state=1).index.values
X = np.concatenate([XC[ci_], np.array(arr[te_i[ni_]])]); src = np.r_[np.zeros(NS), np.ones(NS)].astype(int)

@torch.no_grad()
def feats(model, a, bs=256):
    out = []
    for i in range(0, len(a), bs):
        with torch.autocast("cuda"):
            out.append(model.forward_features(prep(a[i:i + bs])).float().mean((2, 3)).cpu())
    return torch.cat(out).numpy()
def src_auc(Z_):
    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
    p = cross_val_predict(clf, Z_, src, cv=StratifiedKFold(5, shuffle=True, random_state=0), method="predict_proba")[:, 1]
    return roc_auc_score(src, p)
FC, FN = feats(mC, X), feats(mN, X)
Xf = X.astype(np.float32)
stats = np.c_[Xf.mean((1, 2)), Xf.std((1, 2)), Xf[:, :12].mean((1, 2)), Xf[:, -12:].mean((1, 2)), Xf[:, :, :12].mean((1, 2)), Xf[:, :, -12:].mean((1, 2))]
st = pd.DataFrame({"Source-classifier AUROC": {"Brightness/contrast/border statistics": src_auc(stats),
        "Features of CheXpert-trained model": src_auc(FC), "Features of NIH-trained model": src_auc(FN)}})
display(st.round(3)); st.to_csv(rf"{RD}\table_source_classifier_both_models.csv")
fig, ax = plt.subplots(1, 2, figsize=(COL2, 2.6))
for a_, F_, ttl in [(ax[0], FC, "CheXpert-trained features"), (ax[1], FN, "NIH-trained features")]:
    Zt = TSNE(2, perplexity=30, init="pca", random_state=0).fit_transform(PCA(50, random_state=0).fit_transform(F_))
    a_.scatter(Zt[:NS, 0], Zt[:NS, 1], s=2, c="0.2", label="CheXpert"); a_.scatter(Zt[NS:, 0], Zt[NS:, 1], s=2, c="tab:orange", label="NIH")
    a_.set(title=ttl, xlabel="t-SNE 1", ylabel="t-SNE 2"); a_.legend(frameon=False, markerscale=4)
fig.tight_layout(); save_fig(fig, "fig_tsne_both_models")

In [ ]:
import json, shutil, datetime
def to_tex(df, caption, label):
    cols = [df.index.name or ""] + [str(c) for c in df.columns]
    s = "\\begin{table}[t]\n\\centering\n\\caption{%s}\n\\label{%s}\n\\small\n\\resizebox{\\columnwidth}{!}{%%\n\\begin{tabular}{l%s}\n\\hline\n" % (caption, label, "c" * len(df.columns))
    s += " & ".join(cols) + " \\\\\n\\hline\n"
    for idx, r in df.iterrows():
        s += " & ".join([str(idx)] + [(f"{v:.3f}" if isinstance(v, float) else str(v)) for v in r.values]) + " \\\\\n"
    s += "\\hline\n\\end{tabular}}\n\\end{table}\n"
    return s.replace("±", r"$\pm$").replace("→", r"$\rightarrow$").replace("−", "-")
tex = to_tex(mat, "Mean test AUROC over six findings (mean $\\pm$ std over three seeds). Rows: training set; columns: test set.", "tab:transfer") + "\n" \
    + to_tex(perlab, "Per-finding test AUROC for all four train/test combinations.", "tab:four_dirs") + "\n" \
    + to_tex(gaps.drop(columns=["Forward CI", "Reverse CI", "Asymmetry CI"]).set_index("Label").round(3), "AUROC drop from in-hospital to external testing.", "tab:gaps") + "\n" \
    + to_tex(ece_tab, "Expected calibration error.", "tab:ece")
open(rf"{RD}\tables_ieee.tex", "w", encoding="utf-8").write(tex)

BUN = rf"{OUT}\FINAL_bundle"; os.makedirs(BUN, exist_ok=True)
skip = shutil.ignore_patterns("*_last.pt", "*.u8", "__pycache__")
for name, src_ in [("reverse_nih_trained", RD), ("forward_nih_eval", rf"{OUT}\nih_eval"), ("chexpert_combined", rf"{OUT}\combined_e{EPOCHS}")]:
    if os.path.isdir(src_): shutil.copytree(src_, rf"{BUN}\{name}", dirs_exist_ok=True, ignore=skip)
os.makedirs(rf"{BUN}\model_weights", exist_ok=True)
for s in SR:
    for f_ in [rf"{OUT}\{MODEL}_e{EPOCHS}_seed{s}.pt", rf"{RD}\{MODEL}_nih_e{EPOCHS}_seed{s}.pt"]:
        if os.path.exists(f_): shutil.copy(f_, rf"{BUN}\model_weights")
for f_ in [r"G:\chexpert_final_clean.csv", r"G:\nih_split.csv"]:
    if os.path.exists(f_): shutil.copy(f_, BUN)

info = dict(date=str(datetime.date.today()), model=MODEL, epochs=EPOCHS, batch_size=BS, lr=LR, seeds=SR, image_size=224,
            optimizer="AdamW (wd 1e-4), cosine schedule", augmentation="random horizontal flip", init="ImageNet",
            nih_split=dict(train=len(tr_i), val=len(va_i), test=len(te_i)), cheXpert_test=len(cxt), labels=LABELS,
            bootstrap="patient-level, %d resamples" % B, torch=torch.__version__, timm=timm.__version__)
json.dump(info, open(rf"{BUN}\settings.json", "w"), indent=2)
open(rf"{BUN}\README.txt", "w").write(
    "Cross-hospital chest X-ray study: CheXpert and NIH ChestX-ray14, DenseNet121, 3 seeds.\n"
    "reverse_nih_trained/   NIH-trained models: tables, figures, predictions (*_preds.npz), tables_ieee.tex\n"
    "forward_nih_eval/      CheXpert-trained models tested on NIH\n"
    "chexpert_combined/     CheXpert baseline tables, figures, fairness\n"
    "model_weights/         best checkpoints for every seed and direction\n"
    "settings.json          training settings and library versions\n"
    "chexpert_final_clean.csv, nih_split.csv   fixed splits (never change)\n")
zp = shutil.make_archive(rf"{OUT}\FINAL_bundle", "zip", BUN)
print("bundle folder:", BUN); print("zip file:", zp, f"({os.path.getsize(zp) / 1e6:.0f} MB)")